# Format

In [1]:
import numpy as np
import json
import pandas as pd

In [2]:
# DP2, 2024 data
data_df = pd.read_csv(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2024\PUMA\1_yr_ACS_2024_PUMA_NYC_DP02.csv")
data_df

,DP02_0001E,DP02_0001EA,DP02_0001M,DP02_0001MA,DP02_0001PE,DP02_0001PEA,DP02_0001PM,DP02_0001PMA,DP02_0002E,DP02_0002EA,...,DP02_0154MA,DP02_0154PE,DP02_0154PEA,DP02_0154PM,DP02_0154PMA,GEO_ID,NAME,state,public use microdata area,vintage
0,77199,NaN,5893,NaN,77199,NaN,-888888888,NaN,17291,NaN,...,NaN,92.6,NaN,2.5,NaN,NaN,NYC-Manhattan Community District 3--Lower East...,36,4103,2024
1,70182,NaN,5711,NaN,70182,NaN,-888888888,NaN,12520,NaN,...,NaN,93.6,NaN,2.3,NaN,NaN,NYC-Manhattan Community District 4--Chelsea & ...,36,4104,2024
2,114790,NaN,6478,NaN,114790,NaN,-888888888,NaN,46084,NaN,...,NaN,96.2,NaN,1.5,NaN,NaN,NYC-Manhattan Community District 7--Upper West...,36,4107,2024
3,117332,NaN,6065,NaN,117332,NaN,-888888888,NaN,41134,NaN,...,NaN,96.8,NaN,1.4,NaN,NaN,NYC-Manhattan Community District 8--Upper East...,36,4108,2024
4,44279,NaN,3675,NaN,44279,NaN,-888888888,NaN,11736,NaN,...,NaN,92.1,NaN,3.3,NaN,NaN,NYC-Manhattan Community District 9--Morningsid...,36,4109,2024
5,55277,NaN,4596,NaN,55277,NaN,-888888888,NaN,9412,NaN,...,NaN,93.1,NaN,2.9,NaN,NaN,NYC-Manhattan Community District 10--Harlem PU...,36,4110,2024
6,58285,NaN,4459,NaN,58285,NaN,-888888888,NaN,10280,NaN,...,NaN,90.9,NaN,3.3,NaN,NaN,NYC-Manhattan Community District 11--East Harl...,36,4111,2024
7,75119,NaN,4808,NaN,75119,NaN,-888888888,NaN,18192,NaN,...,NaN,93.8,NaN,2.4,NaN,NaN,NYC-Manhattan Community District 12--Washingto...,36,4112,2024
8,79727,NaN,6051,NaN,79727,NaN,-888888888,NaN,23644,NaN,...,NaN,96.6,NaN,1.7,NaN,NaN,NYC-Manhattan Community Districts 1 & 2--Finan...,36,4121,2024
9,114893,NaN,6697,NaN,114893,NaN,-888888888,NaN,27931,NaN,...,NaN,97.1,NaN,1.4,NaN,NaN,NYC-Manhattan Community Districts 5 & 6--Midto...,36,4165,2024


In [3]:
# labels
with open(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\HTM Projects\ACS_Warehouse\docs\dp_variables_2024.json", 'r') as f:
    data = json.load(f)

meta_df = (
    pd.DataFrame.from_dict(data["variables"], orient="index")
      .reset_index()
      .rename(columns={"index": "variable"})
)
meta_df

,variable,label,concept,predicateType,group,limit,predicateOnly,hasGeoCollectionSupport,attributes,required
0,for,Census API FIPS 'for' clause,Census API Geography Specification,fips-for,N/A,0,True,NaN,NaN,NaN
1,in,Census API FIPS 'in' clause,Census API Geography Specification,fips-in,N/A,0,True,NaN,NaN,NaN
2,ucgid,Uniform Census Geography Identifier clause,Census API Geography Specification,ucgid,N/A,0,True,True,NaN,NaN
3,DP02_0126E,Estimate!!ANCESTRY!!Total population!!Arab,Selected Social Characteristics in the United ...,int,DP02,0,NaN,NaN,"DP02_0126EA,DP02_0126M,DP02_0126MA",NaN
4,DP05_0050PE,Percent!!RACE!!Total population!!One race!!Bla...,ACS Demographic and Housing Estimates,float,DP05,0,NaN,NaN,"DP05_0050PEA,DP05_0050PM,DP05_0050PMA",NaN
...,...,...,...,...,...,...,...,...,...,...
1412,DP03_0039PE,Percent!!INDUSTRY!!Civilian employed populatio...,Selected Economic Characteristics,float,DP03,0,NaN,NaN,"DP03_0039PEA,DP03_0039PM,DP03_0039PMA",NaN
1413,DP02_0098E,Estimate!!YEAR OF ENTRY!!Population born outsi...,Selected Social Characteristics in the United ...,int,DP02,0,NaN,NaN,"DP02_0098EA,DP02_0098M,DP02_0098MA",NaN
1414,DP04_0095PE,Percent!!SELECTED MONTHLY OWNER COSTS (SMOC)!!...,Selected Housing Characteristics,float,DP04,0,NaN,NaN,"DP04_0095PEA,DP04_0095PM,DP04_0095PMA",NaN
1415,DP02_0036PE,Percent!!MARITAL STATUS!!Females 15 years and ...,Selected Social Characteristics in the United ...,float,DP02,0,NaN,NaN,"DP02_0036PEA,DP02_0036PM,DP02_0036PMA",NaN


In [11]:
# Define Function to format tables
def acs_dp_to_wide(data_df:pd.DataFrame, meta_df:pd.DataFrame, sample:str, vintage:int) -> pd.DataFrame:
    """
    Function is designed to work with data profile coming from census API requests
    
    Args:
        df_api: Data Profile DataFrame 
        df_meta: Metadata DataFrame
        sample: 1_yr or 5_yr
        vintage: survey year (or last year in 5_yr samples)
    
    Returns:
        pd.DataFrame: Wide format table (variables in rows, geographies in columns) with data profile data

    """
    # PREPARE DATA
    # Remove unnecessary columns from data df
    var_df = data_df.loc[:, ~data_df.columns.str.endswith(('EA', 'MA', 'PEA'))].copy()      # drop annotation columns
    var_df = var_df.drop(columns = ['GEO_ID', 'state', 'vintage', 'NAME'])                  # drop geo_id column (it's empty), state (non-necessary), vintage (useful but makes transpose more difficult),NAME (useful, but messes up the column type)
    
    # Transpose dataframe
    wide_df = var_df.transpose()
    headers = wide_df.iloc[-1].astype(int)                                          # type int to remove decimal point
    wide_df.columns = headers
    wide_df = wide_df[:-1].copy()                                                   # Remove row with headers
    wide_df = wide_df.add_prefix('puma_')
    wide_df = wide_df.reset_index().rename(columns={'index': 'variable'})           # reset index

    # Reincorporate vintage
    wide_df['vintage'] = vintage

    # Add sample indicator
    wide_df['sample'] = sample
    
    # Get variable base name
    wide_df['var_name_base'] = wide_df['variable'].str[0:9]

    # Name qualifier / variable type
    wide_df['var_type'] = wide_df['variable'].str.extract(r'\d+([A-Z]+)', expand=False)
    
    # Change description in var type
    wide_df['var_type'] = np.where(wide_df['var_type'] == 'E', 'Estimate',
                                np.where(wide_df['var_type'] == 'M', 'Estimate MOE',
                                            np.where(wide_df['var_type'] == 'PE', 'Percentage',
                                                     np.where(wide_df['var_type'] == 'PM', 'Percentage MOE', np.nan))))
    
    # PREPARE META DATA
    # Sort values
    meta_df = meta_df.sort_values(by = 'variable')
    
    # Keep only variables in groups DO02 to DP05
    meta_var_df = meta_df[meta_df['group'].isin(['DP02', 'DP03', 'DP04', 'DP05'])].copy()
    
    # Variable base name
    meta_var_df['var_name_base'] = meta_var_df['variable'].str[0:9]

    # Name qualifier / var type
    meta_var_df['var_type'] = meta_var_df['variable'].str.extract(r'\d+([A-Z]+)', expand=False)
    
    # Gen new label column without 'Estimate' or 'Percent' prefix
    meta_var_df['base_label'] = meta_var_df['label'].replace('Estimate|Percent', '', regex=True)
    
    # Keep only the columns we need
    meta_labels_df = meta_var_df[['var_name_base', 'base_label', 'group']].copy()
    
    # Drop duplicates
    meta_labels_df = meta_labels_df.drop_duplicates()
    
    # Label format: Remove initial '!!'
    meta_labels_df['base_label'] = meta_labels_df['base_label'].str.lstrip('!!')

    # Label format: Replace '!!' with ' - '
    meta_labels_df['base_label'] = meta_labels_df['base_label'].str.replace('!!', ' - ', regex=True)

    # Label format: Add a column for the topic of the variable (could be useful when applying filters in Excel)
    meta_labels_df['var_topic'] = meta_labels_df['base_label'].str.split('-').str[0]
    
    
    # MERGE DATA WITH LABELS
    # Merge
    final_df = pd.merge(wide_df, meta_labels_df, how = 'left', left_on='var_name_base', right_on='var_name_base')
    
    # FORMAT FINAL DATA FRAME
    # Drop unnecessary columns
    final_df = final_df.drop(columns = ['var_name_base'])               # Used only for merge
    # rename columns
    final_df = final_df.rename(columns = {'base_label' : 'var_label', 'group' : 'var_group'})
    # sort columns
    final_df = final_df.sort_index(axis = 1, ascending = False)
    
    return(final_df)  

In [10]:
dp_wide = acs_dp_to_wide(data_df=data_df, meta_df=meta_df, sample='1_yr', vintage=2024)

,vintage,variable,var_type,var_topic,var_label,var_group,sample,puma_4503,puma_4502,puma_4501,...,puma_4165,puma_4121,puma_4112,puma_4111,puma_4110,puma_4109,puma_4108,puma_4107,puma_4104,puma_4103
0,2024,DP02_0001E,Estimate,HOUSEHOLDS BY TYPE,HOUSEHOLDS BY TYPE - Total households,DP02,1_yr,59552.0,47171.0,65546.0,...,114893.0,79727.0,75119.0,58285.0,55277.0,44279.0,117332.0,114790.0,70182.0,77199.0
1,2024,DP02_0001M,Estimate MOE,HOUSEHOLDS BY TYPE,HOUSEHOLDS BY TYPE - Total households,DP02,1_yr,3610.0,3115.0,3411.0,...,6697.0,6051.0,4808.0,4459.0,4596.0,3675.0,6065.0,6478.0,5711.0,5893.0
2,2024,DP02_0001PE,Percentage,HOUSEHOLDS BY TYPE,HOUSEHOLDS BY TYPE - Total households,DP02,1_yr,59552.0,47171.0,65546.0,...,114893.0,79727.0,75119.0,58285.0,55277.0,44279.0,117332.0,114790.0,70182.0,77199.0
3,2024,DP02_0001PM,Percentage MOE,HOUSEHOLDS BY TYPE,HOUSEHOLDS BY TYPE - Total households,DP02,1_yr,-888888888.0,-888888888.0,-888888888.0,...,-888888888.0,-888888888.0,-888888888.0,-888888888.0,-888888888.0,-888888888.0,-888888888.0,-888888888.0,-888888888.0,-888888888.0
4,2024,DP02_0002E,Estimate,HOUSEHOLDS BY TYPE,HOUSEHOLDS BY TYPE - Total households - Marrie...,DP02,1_yr,33458.0,24988.0,26486.0,...,27931.0,23644.0,18192.0,10280.0,9412.0,11736.0,41134.0,46084.0,12520.0,17291.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
611,2024,DP02_0153PM,Percentage MOE,COMPUTERS AND INTERNET USE,COMPUTERS AND INTERNET USE - Total households ...,DP02,1_yr,1.8,1.7,1.2,...,0.6,0.4,2.4,2.3,2.2,2.4,0.7,0.9,2.0,2.3
612,2024,DP02_0154E,Estimate,COMPUTERS AND INTERNET USE,COMPUTERS AND INTERNET USE - Total households ...,DP02,1_yr,57576.0,42800.0,61573.0,...,111592.0,77012.0,70483.0,52960.0,51438.0,40771.0,113599.0,110419.0,65713.0,71521.0
613,2024,DP02_0154M,Estimate MOE,COMPUTERS AND INTERNET USE,COMPUTERS AND INTERNET USE - Total households ...,DP02,1_yr,3602.0,2899.0,3569.0,...,6275.0,5811.0,4496.0,3776.0,4550.0,3961.0,6220.0,6232.0,5363.0,6012.0
614,2024,DP02_0154PE,Percentage,COMPUTERS AND INTERNET USE,COMPUTERS AND INTERNET USE - Total households ...,DP02,1_yr,96.7,90.7,93.9,...,97.1,96.6,93.8,90.9,93.1,92.1,96.8,96.2,93.6,92.6


In [34]:
# Save to Excel
dp_wide.to_excel(r"C:\Users\HerreraMartinezR\OneDrive - NYC Office of Management and Budget\Data\ACS\1_yr_sample\2024\PUMA\1_yr_ACS_2024_PUMA_NYC_DP02_WIDE.xlsx", index=False)